# 01 — Esplorazione e pulizia
Controlliamo i tre Parquet TLC di luglio 2025 in `data/raw/`. I timestamp vengono trattati come orari civili di New York con `TIMESTAMP_NTZ`; la sessione Spark usa UTC per preservare i valori di calendario anche quando la lettura Parquet restituisce un timestamp standard.

In [ ]:
from src.config import raw_paths, spark_session
from src.preprocessing import preprocess

paths = raw_paths()
spark = spark_session("TLC 01 - Preprocessing")
report = preprocess(spark, paths)

## Metadati e conteggi
I conteggi per regola si sovrappongono. `removed_unique` conta ogni riga esclusa una sola volta. I delta confrontano la misurazione Spark con la pre-analisi. Per eventuali scostamenti, confrontare prima le regole coinvolte, quindi il tipo dei timestamp, i valori non finiti e la precisione numerica. Il report JSON conserva metadati, soglie e diagnostica.

In [ ]:
from IPython.display import Markdown, display

lines = [
    "| Categoria | Record | Colonne | MiB | Rimasti | Scartati | Esclusi (%) |",
    "|---|---:|---:|---:|---:|---:|---:|",
]
for category, item in report["categories"].items():
    counts = item["counts"]
    lines.append(
        f"| {category} | {counts['original_count']:,} | {len(item['schema'])} "
        f"| {item['size_bytes'] / 2**20:.2f} | {counts['valid_count']:,} "
        f"| {counts['removed_unique']:,} | {item['removed_pct']:.2f} |"
    )
display(Markdown("\n".join(lines)))

for category, item in report["categories"].items():
    print(f"{category}: {item['schema']}")
    print("Conteggi:", item["counts"])
    print("Delta dalla pre-analisi:", item["delta_from_baseline"])

In [ ]:
spark.stop()